Next_Word_Prediction

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\datasets\qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [5]:
quotes = quotes.str.lower()

In [6]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [7]:
quotes

0       “the world as we have created it is a process ...
1       “it is our choices harry that show what we tru...
2       “there are only two ways to live your life one...
3       “the person be it gentleman or lady who has no...
4       “imperfection is beauty madness is genius and ...
                              ...                        
3033         the past beats inside me like a second heart
3034    damn claire warn a guy before you do a facepla...
3035    can you be a girl for a few secondsim always a...
3036    thats what fiction is for its for getting at t...
3037    if we have no peace it is because we have forg...
Name: quote, Length: 3038, dtype: str

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [9]:
vocab_size = 8978

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [10]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [11]:
sequence = tokenizer.texts_to_sequences(quotes)

In [12]:
sequence

[[713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752,
  70,
  2461],
 [947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676],
 [1337,
  14,
  53,
  201,
  714,
  3,
  81,
  15,
  36,
  37,
  7,
  29,
  329,
  93,
  7,
  5,
  1157,
  1,
  101,
  7,
  29,
  329,
  126,
  7,
  5,
  3677],
 [713,
  116,
  12,
  10,
  2462,
  32,
  1043,
  30,
  82,
  13,
  601,
  11,
  5,
  74,
  1338,
  119,
  12,
  2463,
  3678],
 [3679, 7, 313, 753, 7, 638, 4, 43, 144, 3, 12, 682, 1339, 54, 682, 3680],
 [3681, 13, 3, 202, 5, 90, 8, 434, 279, 202, 5, 90, 8, 3682],
 [947,
  7,
  144,
  3,
  12,
  1340,
  17,
  21,
  2,
  14,
  54,
  3,
  12,
  175,
  17,
  21,
  2,
  14,
  3683],
 [715, 16, 13, 1341, 191, 51, 415, 2464, 714, 9, 363, 3684],
 [1044,
  180,
  7,
  39,
  5,
  810,
  1342,
  2,
  46,
  50,
  59,
  322,
  10,
  7,
  168,
  43,
  11,
  639,
  3685],
 [1044, 111, 104, 1045, 7, 39, 2, 50, 3686],
 [2465,
  36,
  7,
  

In [13]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [14]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [15]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [16]:
len(X)

85270

In [17]:
len(y)

85270

In [18]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [19]:
from keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [20]:
y = np.array(y)

In [21]:
X_padded.shape

(85270, 745)

In [22]:
from keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [23]:
y_one_hot.shape

(85270, 8978)

In [24]:
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [25]:
embedding_dim = 50
rnn_units = 128

In [26]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [27]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [28]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [30]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
epochs = 100
batch_size = 128

In [33]:
from keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [34]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [35]:
from keras.preprocessing.sequence import pad_sequences

In [36]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=max_len,padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [37]:
seed_text = "all the"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

secrets


In [38]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [39]:
seed = "the meaning of life"
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

the meaning of life is going to mess up your own and be indulgent


In [40]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [41]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)